## Revision_Robustness Test
## Label Flipping (noise rate 0%, 10%, 20%, 30%)

In [ ]:
from sklearn import datasets
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import confusion_matrix
import math
from keras.models import Sequential
from keras.layers import Dense
from tensorflow.keras.optimizers import SGD, Adam
from matplotlib import pyplot as plt
from sklearn import metrics
import tensorflow as tf
import keras
from keras.layers import BatchNormalization
from keras.layers import Activation
from keras import optimizers
from keras import initializers

## Check if there Null data

In [ ]:
for i in range(1, 106):
    df = pd.read_csv(r'/afs/crc.nd.edu/user/d/dhan6/3. Loss Function/'+'ds'+ str(i) +'.csv')
    print('='*50)
    print('shape of {}:'.format(i), df.shape)
    print(df.columns)                            # features names and label
    if df.isnull().sum().sum() != 0:             # null?
        print(df.isnull().sum())
        break

## Definitions for Experiments

In [ ]:
# To avoid learning fail (log or denominator)
s = 1e-5

import pandas as pd
from sklearn import metrics
import tensorflow as tf
################################ MSE ################################
def MSE(y_true, y_pred):
    return tf.reduce_mean(tf.math.square(y_true-y_pred))

################################ BCE ################################
def BCE(y_true, y_pred):
    return -tf.reduce_mean(y_true*tf.math.log(y_pred+s)+(1-y_true)*tf.math.log(1-y_pred+s))

################################ WBCE ################################
def WBCE(y_true, y_pred):
    N = batch    # batch_size
    y1 = tf.reduce_sum(y_true)
    y0 = N-y1
    w1 = y0/N #N/y1
    w0 = y1/N #N/y0
    return -tf.reduce_mean(w1*y_true*tf.math.log(y_pred+s)+w0*(1-y_true)*tf.math.log(1-y_pred+s))

################################ TN/FP/FN/TP ################################
def confusion_matrix(y_true, y_pred):
    N = batch    # batch_size
    y1 = tf.reduce_sum(y_true)
    y0 = N-y1
    TN = N-tf.reduce_sum(y_true)-tf.reduce_sum(y_pred)+tf.reduce_sum(y_true*y_pred)
    FP = tf.reduce_sum(y_pred)-tf.reduce_sum(y_true*y_pred)
    FN = tf.reduce_sum(y_true)-tf.reduce_sum(y_true*y_pred)
    TP = tf.reduce_sum(y_true*y_pred)
    return N, y1, y0, TN, FP, FN, TP

#################################### Common Functions ####################################
def get_results(y, predicted):
    if np.isnan(predicted).any():
        acc = 0
        pre = 0
        rec = 0
        spe = 0
        f1 = 0
        f05 = 0
        f2 = 0
        gmean = 0
        bacc = 0
    else:
#         print("Conf. Mat:\n", pd.DataFrame(metrics.confusion_matrix(y, predicted)).rename(index={0:'Real(-1/0)', 1:'Real(1)'}, columns={0:'Pred(-1/0)', 1:'Pred(1)'}))  
        TN = metrics.confusion_matrix(y, predicted)[0,0]
        FP = metrics.confusion_matrix(y, predicted)[0,1]
        FN = metrics.confusion_matrix(y, predicted)[1,0]
        TP = metrics.confusion_matrix(y, predicted)[1,1]
#         print("TN, FP, FN, TP:", TN, FP, FN, TP)
        acc = np.round((TP+TN)/(TP+TN+FP+FN),4)
        if TP+FP == 0:
            pre = 0
        else:
            pre = np.round(TP/(TP+FP),4)
        rec = np.round(TP/(TP+FN),4)
        spe = np.round(TN/(FP+TN),4)
        f1 = np.round(TP/(TP + 0.5*(FP+FN)),4)
        f05 = np.round(TP/(TP + 0.8*FP + 0.2*FN),4)
        f2 = np.round(TP/(TP + 0.2*FP + 0.8*FN),4)
        gmean = np.round(((TP/(TP+FN)) * (TN/(TN+FP)))**0.5,4)
        bacc = np.round(0.5*(TP/(TP+FN) + TN/(TN+FP)),4)
    
    return acc, pre, rec, spe, f1, f05, f2, gmean, bacc

# =================================== Fbeta =================================== #

################################ Any_Fbeta ################################
def Any_Fbeta(y_true, y_pred):
    b = 1 
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    F_beta = ((1+b**2)*TP) / ((b**2)*y1 + tf.reduce_sum(y_pred)+s)  # (1+b**2)TP/((1+b**2)TP+FP+b**2FN)
    return 1-F_beta

################################ WBCEFL ################################
def WBCEFL(y_true, y_pred):
    r = 0.7
    b = 1
    WBCEloss = WBCE(y_true, y_pred)
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    F_beta = ((1+b**2)*TP) / ((b**2)*y1 + tf.reduce_sum(y_pred)+s)  # (1+b**2)TP/((1+b**2)TP+FP+b**2FN)
    return (1-r)*WBCEloss+(r)*(1-F_beta)

# =================================== Gmean =================================== #

################################ Any_Gmean ################################
def Any_Gmean(y_true, y_pred):
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    sur_gmean = (TP*TN)/(y1*y0+s)
    return 1-sur_gmean

################################ WBCEGL ################################
def WBCEGL(y_true, y_pred):
    r = 0.3
    WBCEloss = WBCE(y_true, y_pred)
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    sur_gmean = (TP*TN)/(y1*y0+s)
    return (1-r)*WBCEloss+(r)*(1-sur_gmean)

# =================================== BAccu =================================== #

################################ Any_BAccu ################################
def Any_BAccu(y_true, y_pred):
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    baccu = (y0*TP+y1*TN) / (2*y1*y0+s)
    return 1-baccu

################################ WBCEBL ################################
def WBCEBL(y_true, y_pred):
    r = 0.1
    WBCEloss = WBCE(y_true, y_pred)
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    baccu = (y0*TP+y1*TN) / (2*y1*y0+s)
    return (1-r)*WBCEloss+(r)*(1-baccu)

In [ ]:
hidden_node = 2
activation = 'sigmoid'  
kernel_initializer=keras.initializers.he_normal(seed=100)
epochs = 50
threshold = 0.5
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state = 2)
L = 30
learning_rate = 0.01

In [ ]:
# Example pseudo-code
def inject_label_noise(y, noise_ratio, seed=None):
    np.random.seed(seed)
    n = len(y)
    n_flip = int(noise_ratio * n)
    flip_indices = np.random.choice(n, size=n_flip, replace=False)
    y_noisy = y.copy()
    y_noisy[flip_indices] = 1 - y_noisy[flip_indices]  # flip 0↔1
    return y_noisy

In [ ]:
noise_ratio = 0.2    # 0, 0.1, 0.2, 0.3        \\\    y_noise = inject_label_noise(y, noise_ratio, seed=100)

# Experiments for 105-3 Datasets
for i in range(1, 106):
    if i == 23 or i == 82 or i == 84:
        continue
    df = pd.read_csv(r'/afs/crc.nd.edu/user/d/dhan6/3. Loss Function/'+'ds'+ str(i) +'.csv')
    print('+'*35, '{}th Dataset'.format(i), '+'*35)
    print('<Original Class>\n', df.iloc[:,-1].value_counts())
    
    # Make major class as '0' and minor class as '1'
    MAJOR = df.iloc[:,-1].value_counts()[df.iloc[:,-1].value_counts() == max(df.iloc[:,-1].value_counts())].index[0]
    minor = df.iloc[:,-1].value_counts()[df.iloc[:,-1].value_counts() != max(df.iloc[:,-1].value_counts())].index[0]
    df.iloc[:,-1] = df.iloc[:,-1].replace(MAJOR, -100)
    df.iloc[:,-1] = df.iloc[:,-1].replace(minor, 1)
    df.iloc[:,-1] = df.iloc[:,-1].replace(-100, 0)
    print('<Modified Class>\n', df.iloc[:,-1].value_counts())
    print('<Imabalance ratio>\n', "{: .2f}:1".format(df.iloc[:,-1].value_counts()[0]/df.iloc[:,-1].value_counts()[1]))
    
    X = df.iloc[:, :-1]
    X = (X - X.mean())/X.std()    # Features // Standardization
    y = df.iloc[:, -1]
       
    res = pd.DataFrame({'Dataset':['D', 'S', 0, 'C', 'S', 'V', ' ']}, 
                       index = ['Acc','F1','Gmean','B_Acc','Pre','Rec','Spe'])
    res.iloc[2,0] = i    
            
    ##################### For Loop for Every Loss Functions #######################
    LossFunction = [MSE, BCE, WBCE,
                    Any_Fbeta, WBCEFL,
                    Any_Gmean, WBCEGL,
                    Any_BAccu, WBCEBL]
    LossName = ['MSE', 'BCE', 'WBCE',
                'Any_Fbeta', 'WBCEFL',
                'Any_Gmean', 'WBCEGL',
                'Any_BAccu', 'WBCEBL']
    
    for k in range(len(LossFunction)):
        print("={}=".format(LossName[k]))
        repeat_acc = []
        repeat_f1 = []
        repeat_gmean = []
        repeat_bacc = []
        repeat_pre = []
        repeat_rec = []
        repeat_spe = []
        for t in range(1):     # 5 times repeat
#             print("=========="*3, "{}th repeat".format(t+1), "=========="*3)
            list_acc = []
            list_f1 = []
            list_gmean = []
            list_bacc = []
            list_pre = []
            list_rec = []
            list_spe = []
            n_iter=0
            for train_index, test_index in skf.split(df, df.iloc[:,-1]):
                n_iter += 1
                X_train = X.iloc[train_index]
                y_train= y.iloc[train_index]
                X_test = X.iloc[test_index]
                y_test= y.iloc[test_index]
#                 print('#'*10,'{0}th CV'.format(n_iter),'#'*10)
                X_train = np.array(X_train)
                y_train = np.array(y_train)
                y_noise = inject_label_noise(y_train, noise_ratio, seed=100) ## noise injection
                y_train = y_noise.astype(float)
                X_test = np.array(X_test)
                y_test = np.array(y_test)
                y_test = y_test.astype(float)
                batch = int(X_train.shape[0] * 0.05)
#                 if k in [5, 6, 9, 10, 13, 14, 17, 18]:
#                     learning_rate = 0.02
#                     batch = int(np.ceil(X_train.shape[0] * 0.5))
#                 if k in [11, 12]:
#                     learning_rate = 0.1
#                     batch = int(np.ceil(X_train.shape[0] * 0.5))
                model = Sequential()
                model.add(Dense(hidden_node, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
                model.add(BatchNormalization())
                model.add(Activation(activation))
                model.add(Dense(1, activation='sigmoid'))
                opt = tf.keras.optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)
                model.compile(loss=LossFunction[k], optimizer=opt, metrics=['accuracy'])
                history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch)        
#                 plt.plot(history.history['loss'], label='loss')
#                 plt.ylim([0, 1])
#                 plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#                 plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#                 plt.title("Cost Function",fontweight="bold",fontsize = 20)
#                 plt.legend()
#                 plt.show() 
                result = model.predict(X_test, verbose=0)
                predicted = np.round(result)
                acc, pre, rec, spe, f1, f05, f2, gmean, bacc = get_results(y_test, predicted)
                # Cross Validation Results
                list_acc.append(acc)
                list_f1.append(f1)
                list_gmean.append(gmean)
                list_bacc.append(bacc)
                list_pre.append(pre)
                list_rec.append(rec)
                list_spe.append(spe)
            # Cross Validation Results
            repeat_acc.append(np.mean(list_acc))
            repeat_f1.append(np.mean(list_f1))
            repeat_gmean.append(np.mean(list_gmean))
            repeat_bacc.append(np.mean(list_bacc))
            repeat_pre.append(np.mean(list_pre))
            repeat_rec.append(np.mean(list_rec))
            repeat_spe.append(np.mean(list_spe))

        res['{}'.format(LossName[k])] = [np.mean(repeat_acc), np.mean(repeat_f1), np.mean(repeat_gmean), np.mean(repeat_bacc),
                                         np.mean(repeat_pre), np.mean(repeat_rec), np.mean(repeat_spe)] 
    
    res.to_csv("[REV-robustness]5CV_MLP(AllTypes_1Repeat)_noise_2.csv", mode = 'a', float_format='%.4g')

In [ ]:
res